# 🚀 Konversi Dataset Phishing: `vonDataset20180426.dill` ke Format CSV

Notebook ini berjalan langsung di **Google Colab** untuk:
1. Otomatis me-mount **Google Drive**.
2. Membuat folder target `/content/drive/MyDrive/Skripsi/new data` jika belum ada.
3. Membaca dataset serialized dari `/content/drive/MyDrive/Skripsi/vonDataset20180426.dill`.
4. Mendekode kembali representasi karakter integer menjadi string URL asli.
5. Menyimpan file hasil konversi ke format `.csv` (`train.csv`, `val.csv`, `test.csv`, dan `vonDataset20180426.csv`) di dalam folder **`new data`**.

## 1. Install Library Pendukung

In [1]:
# Install dill untuk membaca file serialisasi .dill
!pip install dill pandas numpy tqdm -q

import os
import gc
import time
import dill
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

print("[OK] Semua library berhasil di-import.")

[OK] Semua library berhasil di-import.


## 2. Mount Google Drive & Pembuatan Folder `new data`

In [4]:
# 1. Me-mount Google Drive
print("[INFO] Me-mount Google Drive...")
drive.mount('/content/drive')

# 2. Tentukan path file input dan folder output sesuai permintaan
INPUT_FILE = "/content/drive/MyDrive/Skripsi/vonDataset20180426.dill"
OUTPUT_DIR = "/content/drive/MyDrive/Skripsi/new data"

# 3. Buat folder 'new data' jika belum ada
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] Direktori penyimpanan siap: {OUTPUT_DIR}")

# 4. Verifikasi keberadaan file dataset .dill
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"❌ File tidak ditemukan di: {INPUT_FILE}\n"
        f"Harap pastikan file 'vonDataset20180426.dill' telah berada di folder Google Drive Anda: /content/drive/MyDrive/Skripsi/"
    )

size_mb = os.path.getsize(INPUT_FILE) / (1024 * 1024)
print(f"[OK] File input ditemukan: {INPUT_FILE} ({size_mb:.2f} MB)")

Mounted at /content/drive
[OK] Direktori penyimpanan siap: /content/drive/MyDrive/Skripsi/new data
[OK] File input ditemukan: /content/drive/MyDrive/Skripsi/vonDataset20180426.dill (898.22 MB)


## 3. Pemuatan File `.dill` ke Memori

In [5]:
print("[INFO] Membaca file .dill ke memori (membutuhkan waktu ~15-30 detik)... ")
start_t = time.perf_counter()

with open(INPUT_FILE, "rb") as f:
    raw_data = dill.load(f)

elapsed = time.perf_counter() - start_t
print(f"[OK] File berhasil dimuat dalam {elapsed:.2f} detik!")

print("\n=== RINGKASAN STRUKTUR DATASET ===")
for key, val in raw_data.items():
    if hasattr(val, "shape"):
        print(f"- {key:<12}: ndarray shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, dict):
        print(f"- {key:<12}: dictionary ({len(val)} karakter vocabulary)")

[INFO] Membaca file .dill ke memori (membutuhkan waktu ~15-30 detik)... 
[OK] File berhasil dimuat dalam 49.99 detik!

=== RINGKASAN STRUKTUR DATASET ===
- char_to_int : dictionary (100 karakter vocabulary)
- test_y      : ndarray shape=(155937,), dtype=int32
- val_y       : ndarray shape=(155936,), dtype=int32
- train_x     : ndarray shape=(1247489, 150), dtype=int32
- train_y     : ndarray shape=(1247489,), dtype=int32
- test_x      : ndarray shape=(155937, 150), dtype=int32
- val_x       : ndarray shape=(155936, 150), dtype=int32


## 4. Setup Decoder Karakter (Integer Sequence -> URL String)

In [6]:
# Buat pemetaan balik int -> char dari char_to_int vocabulary
char_to_int = raw_data["char_to_int"]
int_to_char = {}
for ch, idx in char_to_int.items():
    if ch not in ('<PAD>', '<pad>', '<nul>', '', '\x00', None):
        int_to_char[idx] = ch

# Buat array lookup berkecepatan tinggi
max_idx = max(int_to_char.keys()) if int_to_char else 256
for split_k in ["train_x", "val_x", "test_x"]:
    if split_k in raw_data:
        max_idx = max(max_idx, int(np.max(raw_data[split_k][:500])))

lookup_size = max_idx + 20
lookup_table = [""] * lookup_size
for idx, ch in int_to_char.items():
    if idx < lookup_size:
        lookup_table[idx] = ch

def decode_url(int_sequence):
    """Mendekode array integer 1D menjadi string URL."""
    return "".join(lookup_table[c] for c in int_sequence if c < lookup_size and lookup_table[c]).strip()

# Uji coba decoding 3 sampel URL data latih
print("[INFO] Uji coba decoding 3 sampel URL:")
for i in range(3):
    sample_url = decode_url(raw_data["train_x"][i])
    label_str = "Phishing (1)" if raw_data["train_y"][i] == 1 else "Benign (0)"
    print(f" [{i+1}] {label_str:<12} -> {sample_url}")

[INFO] Uji coba decoding 3 sampel URL:
 [1] Phishing (1) -> 补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补http://bs-369.com/Views/Account/Login.aspx?ReturnUrl=/
 [2] Phishing (1) -> 补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补http://web.furell.net/web/Login.html
 [3] Phishing (1) -> 补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补http://amelihair.ru/images/6550054213/745353769679/


## 5. Konversi dan Ekspor ke CSV di `/content/drive/MyDrive/Skripsi/new data`

In [7]:
def convert_and_save_csv(X, y, split_name, filename, batch_size=50000):
    """
    Mendekode array per batch dan menyimpan langsung ke CSV di folder 'new data'.
    Menggunakan batching agar menghemat memori RAM Colab.
    """
    output_path = os.path.join(OUTPUT_DIR, filename)
    total_rows = len(y)
    print(f"\n[PROSES] Mengonversi split '{split_name}' ({total_rows:,} baris) -> {output_path}...")
    
    t0 = time.perf_counter()
    is_first = True
    
    for start_i in tqdm(range(0, total_rows, batch_size), desc=f"Export {split_name}"):
        end_i = min(start_i + batch_size, total_rows)
        batch_x = X[start_i:end_i]
        batch_y = y[start_i:end_i]
        
        # Decode string URL
        urls = [decode_url(row) for row in batch_x]
        
        df_batch = pd.DataFrame({
            "url": urls,
            "label": batch_y
        })
        
        if is_first:
            df_batch.to_csv(output_path, index=False, mode="w")
            is_first = False
        else:
            df_batch.to_csv(output_path, index=False, mode="a", header=False)
            
        del urls, df_batch
        
    dur = time.perf_counter() - t0
    file_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[OK] Berhasil disimpan: {output_path}")
    print(f"     Ukuran: {file_mb:.2f} MB | Waktu: {dur:.2f} detik")
    gc.collect()
    return output_path

# 1. Ekspor train.csv (~1.247.489 baris)
train_csv_path = convert_and_save_csv(raw_data["train_x"], raw_data["train_y"], "Train", "train.csv")

# 2. Ekspor val.csv (~155.936 baris)
val_csv_path = convert_and_save_csv(raw_data["val_x"], raw_data["val_y"], "Validation", "val.csv")

# 3. Ekspor test.csv (~155.937 baris)
test_csv_path = convert_and_save_csv(raw_data["test_x"], raw_data["test_y"], "Test", "test.csv")

# 4. Ekspor dataset gabungan utuh: vonDataset20180426.csv (~1.559.362 baris)
combined_csv_path = os.path.join(OUTPUT_DIR, "vonDataset20180426.csv")
print(f"\n[PROSES] Membuat file CSV gabungan lengkap: {combined_csv_path}...")
t_c = time.perf_counter()
first_sp = True

splits = [
    ("train", train_csv_path),
    ("val", val_csv_path),
    ("test", test_csv_path)
]

for split_tag, path_src in splits:
    print(f"[*] Menggabungkan partisi '{split_tag}'...")
    for chunk in pd.read_csv(path_src, chunksize=100000):
        chunk["split"] = split_tag
        if first_sp:
            chunk.to_csv(combined_csv_path, index=False, mode="w")
            first_sp = False
        else:
            chunk.to_csv(combined_csv_path, index=False, mode="a", header=False)

dur_c = time.perf_counter() - t_c
mb_c = os.path.getsize(combined_csv_path) / (1024 * 1024)
print(f"[OK] File CSV gabungan berhasil dibuat: {combined_csv_path}")
print(f"     Ukuran: {mb_c:.2f} MB | Waktu: {dur_c:.2f} detik")
gc.collect()


[PROSES] Mengonversi split 'Train' (1,247,489 baris) -> /content/drive/MyDrive/Skripsi/new data/train.csv...


Export Train:   0%|          | 0/25 [00:00<?, ?it/s]

[OK] Berhasil disimpan: /content/drive/MyDrive/Skripsi/new data/train.csv
     Ukuran: 363.52 MB | Waktu: 70.31 detik

[PROSES] Mengonversi split 'Validation' (155,936 baris) -> /content/drive/MyDrive/Skripsi/new data/val.csv...


Export Validation:   0%|          | 0/4 [00:00<?, ?it/s]

[OK] Berhasil disimpan: /content/drive/MyDrive/Skripsi/new data/val.csv
     Ukuran: 45.40 MB | Waktu: 8.24 detik

[PROSES] Mengonversi split 'Test' (155,937 baris) -> /content/drive/MyDrive/Skripsi/new data/test.csv...


Export Test:   0%|          | 0/4 [00:00<?, ?it/s]

[OK] Berhasil disimpan: /content/drive/MyDrive/Skripsi/new data/test.csv
     Ukuran: 45.44 MB | Waktu: 10.57 detik

[PROSES] Membuat file CSV gabungan lengkap: /content/drive/MyDrive/Skripsi/new data/vonDataset20180426.csv...
[*] Menggabungkan partisi 'train'...
[*] Menggabungkan partisi 'val'...
[*] Menggabungkan partisi 'test'...
[OK] File CSV gabungan berhasil dibuat: /content/drive/MyDrive/Skripsi/new data/vonDataset20180426.csv
     Ukuran: 462.84 MB | Waktu: 28.66 detik


0

## 6. Verifikasi File CSV di Folder Google Drive

In [8]:
print(f"=== DAFTAR FILE HASIL DI: {OUTPUT_DIR} ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".csv"):
        fp = os.path.join(OUTPUT_DIR, f)
        mb = os.path.getsize(fp) / (1024 * 1024)
        print(f"✅ {f:<26} : {mb:6.2f} MB")

print("\n=== PREVIEW 5 BARIS PERTAMA train.csv ===")
display(pd.read_csv(train_csv_path, nrows=5))

print("\n[SELESAI] Konversi selesai dan file CSV telah aman tersimpan di Google Drive! 🎉")

=== DAFTAR FILE HASIL DI: /content/drive/MyDrive/Skripsi/new data ===
✅ test.csv                   :  45.44 MB
✅ train.csv                  : 363.52 MB
✅ val.csv                    :  45.40 MB
✅ vonDataset20180426.csv     : 462.84 MB

=== PREVIEW 5 BARIS PERTAMA train.csv ===


,url,label
0,补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补...,1
1,补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补...,1
2,补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补...,1
3,补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补...,0
4,补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补补...,1



[SELESAI] Konversi selesai dan file CSV telah aman tersimpan di Google Drive! 🎉


In [9]:
# ==============================================================================
# 7. Pembersihan Karakter Padding '补' & Ekspor ke vonDataset_clean.csv
# ==============================================================================
import gc
import os
import pandas as pd
from tqdm.auto import tqdm

# 1. Definisi path dataset sumber dan target
INPUT_CSV = "/content/drive/MyDrive/Skripsi/new data/vonDataset20180426.csv"
OUTPUT_CSV = "/content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv"

# Validasi keberadaan file sumber
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"❌ File tidak ditemukan: {INPUT_CSV}\n"
        f"Pastikan file CSV gabungan dari cell sebelumnya telah selesai diekspor."
    )

print(f"[INFO] Membersihkan padding '补' dari : {INPUT_CSV}")
print(f"[INFO] File hasil akan disimpan ke       : {OUTPUT_CSV}\n")

# Hapus file output lama jika sudah ada agar mode append bersih dari awal
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

# 2. Proses pembersihan data per chunk (100.000 baris per iterasi)
CHUNK_SIZE = 100_000
total_cleaned = 0
label_counts = {0: 0, 1: 0}
first_chunk = True

for chunk in tqdm(
    pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE), desc="Membersihkan URL"
):
    # A. Bersihkan karakter padding '补' dan spasi kosong di awal/akhir URL
    chunk["url"] = (
        chunk["url"].astype(str).str.replace("补", "", regex=False).str.strip()
    )

    # B. Hapus baris yang kosong atau rusak (URL kosong, whitespace saja, atau NaN)
    chunk = chunk[chunk["url"].notna() & (chunk["url"] != "")]

    # C. Validasi dan pastikan label hanya 0 (aman) atau 1 (phishing)
    chunk["label"] = pd.to_numeric(chunk["label"], errors="coerce")
    chunk = chunk[chunk["label"].isin([0, 1])]
    chunk["label"] = chunk["label"].astype(int)

    # Catat akumulasi label
    counts = chunk["label"].value_counts()
    label_counts[0] += counts.get(0, 0)
    label_counts[1] += counts.get(1, 0)

    # D. Simpan ke CSV (mode 'w' untuk baris header pertama, 'a' untuk lanjutannya)
    if first_chunk:
        chunk.to_csv(OUTPUT_CSV, index=False, mode="w")
        first_chunk = False
    else:
        chunk.to_csv(OUTPUT_CSV, index=False, mode="a", header=False)

    total_cleaned += len(chunk)

# Bebaskan memori
gc.collect()

# 3. Tampilkan ringkasan statistik dataset bersih
file_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

print("\n" + "=" * 60)
print(f"✅ [SUKSES] Dataset bersih berhasil disimpan!")
print(f"📁 Path File         : {OUTPUT_CSV}")
print(f"📦 Ukuran File       : {file_mb:.2f} MB")
print(f"📊 Total Baris Bersih: {total_cleaned:,} baris")
print("📈 Distribusi Label  :")
print(
    f"   - Aman (0)       : {label_counts[0]:,} ({label_counts[0] / total_cleaned * 100:.2f}%)"
)
print(
    f"   - Phishing (1)   : {label_counts[1]:,} ({label_counts[1] / total_cleaned * 100:.2f}%)"
)
print("=" * 60)

# 4. Tampilkan 5 baris pertama sebagai pratinjau
print("\n=== PRATINJAU 5 BARIS PERTAMA ===")
df_preview = pd.read_csv(OUTPUT_CSV, nrows=5)
display(df_preview)


[INFO] Membersihkan padding '补' dari : /content/drive/MyDrive/Skripsi/new data/vonDataset20180426.csv
[INFO] File hasil akan disimpan ke       : /content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv



Membersihkan URL: 0it [00:00, ?it/s]


✅ [SUKSES] Dataset bersih berhasil disimpan!
📁 Path File         : /content/drive/MyDrive/Skripsi/new data/vonDataset_clean.csv
📦 Ukuran File       : 122.63 MB
📊 Total Baris Bersih: 1,559,362 baris
📈 Distribusi Label  :
   - Aman (0)       : 800,000 (51.30%)
   - Phishing (1)   : 759,362 (48.70%)

=== PRATINJAU 5 BARIS PERTAMA ===


,url,label,split
0,http://bs-369.com/Views/Account/Login.aspx?Ret...,1,train
1,http://web.furell.net/web/Login.html,1,train
2,http://amelihair.ru/images/6550054213/74535376...,1,train
3,https://www.behance.net/gallery/23251525/Self-...,0,train
4,http://ssl32.sslbr.co.vu/gol/,1,train
